# 22 — Does the revised rubric improve tagging?

**DR-67 · Gemini 3.8 Flash and Qwen3.6 · 54 manually reviewed demonstrations**

DR-65 found that rearranging the old rules did not help. DR-66 then clarified the actual rules and examples. Here we compare **complete v3 against complete v4**, using fresh calls for both. Each model receives the same title and can use its learned knowledge. Search stays off; each answer may contain up to two tags.

We score both versions against the same **54 accepted reviews and 89 reference tags**. No unreviewed historical records enter the calls or scores. These cases helped shape v4, so any improvement is development evidence, not a new final-test result.

Private source files are required to execute this notebook. Public readers can use the [aggregate result summary](../docs/results/rubric-v4-comparison.md). Default execution replays saved responses without API calls.


In [ ]:
import copy
import json
import os
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from threading import Event

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path[:0] = [str(root), str(root / "src")]
from dotenv import load_dotenv
from notebooks.tools import verify_sha256, show_table, fingerprint, percent
from notebooks.tools.hashes import file_sha256
from notebooks.tools.hosted import chat_request
from notebooks.tools.gem_profile import gem_request, PRICING
from notebooks.tools.gemini import decode_result
from notebooks.tools.costs import BudgetGuard, cost_microusd
from demo_logic.notebook_support import save_private_json
from demo_logic.evaluation import multilabel_metrics
from demo_logic.taxonomy import topic_tags

load_dotenv(root / ".env", override=False)
LIVE = os.getenv("NOTEBOOK22_RUN_LIVE", "false").lower() == "true"
config_path = root / "notebooks/prompts/rubric-v4-comparison-v1.json"
config = json.loads(config_path.read_text())
folder = root / "reports/rubric-policy/v1"
run_id = config["version"]

def read_pinned(item):
    """Read an approved input only when its bytes match the saved fingerprint.

    Private references must already exist in this authorized checkout. Changing
    a document requires a new reviewed configuration, not silently updating a hash.
    """
    path = root / item["path"]
    if not path.is_file():
        raise FileNotFoundError(f"Private input unavailable: {item['path']}. Read the public result summary instead.")
    verify_sha256(path, item["sha256"])
    return path.read_text()

reference = json.loads(read_pinned(config["reference"]))
old_reference = json.loads(read_pinned(config["original_reference"]))
reviewed = {r["external_id"]: r for r in reference["records"] if r["score_eligible"]}
old_reviewed = {r["external_id"]: r for r in old_reference["records"] if r["score_eligible"]}
ids = sorted(reviewed)
assert len(ids) == config["reference_count"] == 54 and set(ids) == set(old_reviewed)
assert sum(len(r["reference_tags"]) for r in reviewed.values()) == config["reference_tags"] == 89
assert all(reviewed[i]["title"] == old_reviewed[i]["title"] for i in ids)
documents = {arm: read_pinned(item) for arm, item in config["rubrics"].items()}
tags = sorted(topic_tags("v2"))
models = [c["model"] for c in config["candidates"]]
arms = ["v3", "v4"]
show_table(["Comparison", "Setting"], [("Reviewed demonstrations", len(ids)), ("Reference tags", 89),
    ("Unreviewed historical records", 0), ("Search", "Off; learned knowledge allowed"),
    ("Maximum suggested tags per answer", 2), ("Scheduled requests", 216)])


## 1. Keep the comparison fair

Only the rubric document and its version number change between the two requests. Both use the same response instructions, title, model and generation settings. No reviewed answers, private notes or case IDs are sent. The common instructions make clear that review statuses are not model-output fields.

For each model, half the titles receive v3 first and half receive v4 first. Both arms get fresh calls in their own cache. Gemini keeps its existing provider-default sampling settings; Qwen keeps temperature zero and thinking off. This is one run per condition, so some variation may still come from the provider.

Before seeing results, the screening rule is **at least +3 percentage points F1**, with no loss greater than **2 points in precision or coverage**. We also inspect which tags improve or deteriorate. Project targets remain 80% precision, 80% recall and 95% coverage.


In [ ]:
payloads = {}
for candidate in config["candidates"]:
    model = candidate["model"]
    payloads[model] = {}
    for i in ids:
        payloads[model][i] = {}
        for arm in arms:
            system = config["wrapper"].replace("RUBRIC_VERSION", arm[1:]) + documents[arm]
            title = json.dumps({"title": reviewed[i]["title"]}, ensure_ascii=False)
            p = copy.deepcopy(candidate["settings"])
            if "Qwen" in model:
                p["messages"] = [{"role": "system", "content": system}, {"role": "user", "content": title}]
            else:
                p.update(system_instruction=system, input=title)
            assert not p.get("tools")
            payloads[model][i][arm] = p

# Use the same price record as the audited adapter; prices rechecked on 6 October.
assert PRICING["input"] == config["gemini_prices_usd_per_million"]["input"]
assert PRICING["output"] == config["gemini_prices_usd_per_million"]["output"]
gemini = next(m for m in models if m.startswith("gemini"))
reservation_bounds = {i: {a: cost_microusd(len(json.dumps(p).encode()) + 2048,
    p["generation_config"]["max_output_tokens"], PRICING["input"], PRICING["output"])
    for a, p in pair.items()} for i, pair in payloads[gemini].items()}
batch_bound = sum(sum(pair.values()) for pair in reservation_bounds.values())
assert batch_bound <= config["run_limit_usd"] * 1_000_000
contract = {"config_sha256": file_sha256(config_path), "config": config, "ids": ids,
    "condition_order": "v3 first for even sorted index; v4 first for odd",
    "paid_batch_reservation_microusd": batch_bound,
    "request_fingerprints": {c["model"]: {i: {a: fingerprint({"endpoint": c["endpoint"], "payload": p})
        for a, p in pair.items()} for i, pair in payloads[c["model"]].items()} for c in config["candidates"]}}
contract_path = folder / "contract.json"
if contract_path.exists():
    assert json.loads(contract_path.read_text()) == contract, "Run inputs changed; preserve this run and review a new version."
else:
    save_private_json(contract_path, contract)
print(f"Conservative whole-batch reservation bound: ${batch_bound / 1_000_000:.4f}; run limit: ${config['run_limit_usd']:.2f}.")
print("This is not an invoice. The existing ledger retains earlier spend and unknown-charge reservations.")


## 2. Run or replay saved answers

Leave `NOTEBOOK22_RUN_LIVE=false` in `.env` to replay. Set it to `true` only to run missing requests for this authorized comparison. The private `.env` supplies `GEMINI_API_KEY` and `HETZNER_API_KEY`; keys never enter reports.

The existing ledger checks the $50 project limit, $1 per demonstration across experiments and this run's $5 ceiling before each paid request. Earlier unknown charges retain their reservations. Qwen uses the already-confirmed free Hetzner account. Costs are conservative USD estimates, separate from the euro dashboard.

A transport or accounting problem pauses that provider. Invalid answers remain failures with no tags; we do not repeatedly sample until the model gives a preferred answer. Completed failures are preserved on replay. Original request times are reported, not replay times.


In [ ]:
guard = BudgetGuard(root / "reports/budget/capstone.sqlite", prior_total_microusd=0,
    prior_research_microusd=0, run_limits={run_id: config["run_limit_usd"] * 1_000_000})

def run_model(candidate):
    """Run or replay paired requests while preserving every completed attempt.

    Label data is used by evaluation only. An operational pause stops new live
    calls for this provider; in-flight calls finish and remain auditable. Cached
    failures are not silently retried on a later notebook run.
    """
    model = candidate["model"]
    stopped = Event()
    def run_pair(item):
        index, i = item
        pair = {}
        for arm in (arms if index % 2 == 0 else list(reversed(arms))):
            place = folder / ("qwen" if "Qwen" in model else "gemini") / i / arm
            p = payloads[model][i][arm]
            identity = contract["request_fingerprints"][model][i][arm]
            if "Qwen" in model:
                failed = list((place / "hosted-failures" / identity).glob("*.json"))
                for start in (place / "hosted-attempts").glob("*-started.json"):
                    if not start.with_name(start.name.replace("-started", "-finished")).exists():
                        raise RuntimeError("Interrupted Qwen attempt: inspect before resuming.")
                result = chat_request({**candidate, "key_env": "HETZNER_API_KEY"}, p, tags=tags,
                    cache_dir=place / "cache", allow_live=LIVE and not stopped.is_set() and not failed,
                    api_key=os.getenv("HETZNER_API_KEY"), max_retry_wait_seconds=0)
            else:
                completed = sorted((place / "attempts").glob("*-finished.json"))
                if completed:
                    result = json.loads(completed[-1].read_text())
                    assert result["request_sha256"] == identity
                    if result["status"] == "ok":
                        result.update(decode_result(result["response"], tags))
                else:
                    result = gem_request(p, event_id=i, folder=place, key=os.getenv("GEMINI_API_KEY"),
                        tags=tags, guard=guard, live=LIVE and not stopped.is_set(), decoder=decode_result,
                        run_id=run_id, research_budget=False)
            if result.get("status") == "failed" and result.get("error") == "Invalid or incomplete model response":
                result = {**result, "transport_status": "failed", "status": "invalid_response", "predicted": []}
            if result.get("accounting_pending") or result["status"] not in {"ok", "invalid_response"}:
                stopped.set()
            pair[arm] = {k: v for k, v in result.items() if k != "cached"}
        print(model, index + 1, "/54", {a: x["status"] for a, x in pair.items()}, flush=True)
        return i, pair
    with ThreadPoolExecutor(max_workers=1 if "Qwen" in model else 3) as pool:
        return model, dict(pool.map(run_pair, enumerate(ids)))

with ThreadPoolExecutor(max_workers=2) as pool:
    answers = dict(pool.map(run_model, config["candidates"]))
save_private_json(folder / "progress.json", {"contract": contract, "answers": answers})
complete = all(x["status"] in {"ok", "invalid_response"} and not x.get("accounting_pending")
    for model in models for pair in answers[model].values() for x in pair.values())
print("Paired comparison complete:", complete)


## 3. Did quality improve?

Both rubrics are judged against the **same approved answers**, including the added D6 tag. A second view uses the original answers to show whether that single reference change affects the conclusion.

- **Precision:** how many suggested tags match the reference.
- **Recall:** how many reference tags the model finds.
- **F1:** balances precision and recall.
- **Coverage:** how many demonstrations receive any tag, even if it is wrong.
- **Exact match:** how many demonstrations receive the complete reference tag set.

Disagreement can reflect a model mistake, missing context or a retained human interpretation. The scores alone do not identify the cause.


In [ ]:
if not complete:
    raise RuntimeError("Missing responses or an operational pause. Inspect the private progress report before drawing a paired conclusion.")
metrics = {}
original_metrics = {}
for model in models:
    metrics[model], original_metrics[model] = {}, {}
    for arm in arms:
        predictions = [answers[model][i][arm].get("predicted", []) for i in ids]
        metrics[model][arm] = multilabel_metrics([reviewed[i]["reference_tags"] for i in ids], predictions, tags)
        original_metrics[model][arm] = multilabel_metrics([old_reviewed[i]["reference_tags"] for i in ids], predictions, tags)
show_table(["Model", "Rubric", "Precision", "Recall", "F1", "Coverage", "Exact match"],
    [(m, a, *[percent(metrics[m][a][k]) for k in ["micro_precision", "micro_recall", "micro_f1", "coverage", "exact_match"]])
     for m in models for a in arms])
for model in models:
    before, after = metrics[model]["v3"], metrics[model]["v4"]
    screen = config["screen"]
    passed = (after["micro_f1"] - before["micro_f1"] >= screen["minimum_f1_gain"]
        and after["micro_precision"] - before["micro_precision"] >= -screen["maximum_precision_loss"]
        and after["coverage"] - before["coverage"] >= -screen["maximum_coverage_loss"])
    print(model, "F1 change:", round(100 * (after["micro_f1"] - before["micro_f1"]), 1),
          "percentage points; passes numerical screen:", passed)
show_table(["Model", "Rubric", "F1: revised answers", "F1: original answers"],
    [(m, a, percent(metrics[m][a]["micro_f1"]), percent(original_metrics[m][a]["micro_f1"])) for m in models for a in arms])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
measures = ["micro_precision", "micro_recall", "micro_f1", "coverage"]
x = np.arange(len(measures))
for ax, model in zip(axes, models):
    for offset, arm, color in [(-.18, "v3", "#52677d"), (.18, "v4", "#18a999")]:
        values = [100 * metrics[model][arm][k] for k in measures]
        bars = ax.bar(x + offset, values, .34, label=arm, color=color)
        ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)
    ax.scatter([0, 1, 3], [80, 80, 95], marker="_", color="#cc5933", s=350, label="Targets", zorder=3)
    ax.set_title("Qwen3.6" if "Qwen" in model else "Gemini 3.8 Flash")
    ax.set_xticks(x, ["Precision", "Recall", "F1", "Coverage"])
    ax.set_ylim(0, 112)
axes[0].set_ylabel("Percent")
axes[1].legend(loc="lower right")
fig.suptitle("Fresh v3/v4 calls; same 54 manually reviewed cases")
fig.tight_layout()
display(fig)
plt.close(fig)


## 4. Which tags changed?

The table counts missing reference tags and extra suggestions. Lower is better for each count, but a lower total can hide harm to a particular topic. Education and religion have no reference examples here, so this sample cannot establish recall for those categories.

Five reference answers contain three tags. With a two-tag response, even a perfect model can recover at most **84/89 tags (94.4% recall)**. The private table below shows these five cases separately. A missing third tag may reflect the output limit; we cannot infer that the model internally recognized it.


In [ ]:
for model in models:
    before, after = metrics[model]["v3"], metrics[model]["v4"]
    print(model)
    show_table(["Tag", "Reference count", "Missing v3", "Missing v4", "Extra v3", "Extra v4"],
        [(t, before["per_tag"][t]["support"], before["per_tag"][t]["fn"], after["per_tag"][t]["fn"],
          before["per_tag"][t]["fp"], after["per_tag"][t]["fp"]) for t in tags])
three_tag_ids = [i for i in ids if len(reviewed[i]["reference_tags"]) > 2]
assert len(three_tag_ids) == 5
show_table(["Private case ID", "Reference", "Model", "v3", "v4"],
    [(i, "; ".join(reviewed[i]["reference_tags"]), m,
      "; ".join(answers[m][i]["v3"].get("predicted", [])),
      "; ".join(answers[m][i]["v4"].get("predicted", []))) for i in three_tag_ids for m in models])


## 5. Record results and decide the next step

The saved private report contains requests, answers, fingerprints, original latency, usage, both reference views and per-tag counts. It contains no credentials. Public summaries show aggregates only; clear notebook outputs before committing.

Passing the screen earns consideration for a later confirmation on other manually reviewed cases. It does not prove generalization or justify automatic tagging. If gains do not carry over, stop tuning these same examples and continue to the reviewed prototype/manual-workflow decision. No additional model, search experiment or reference change starts automatically from this notebook.


In [ ]:
from statistics import median

operations = {}
for model in models:
    operations[model] = {}
    for arm in arms:
        results = [answers[model][i][arm] for i in ids]
        durations = [x["duration_seconds"] for x in results if x.get("duration_seconds") is not None]
        costs = [x.get("gross_upper_microusd") for x in results]
        total_cost = 0 if "Qwen" in model else sum(costs) if all(c is not None for c in costs) else None
        operations[model][arm] = {"statuses": dict(Counter(x["status"] for x in results)),
            "median_seconds": median(durations) if durations else None,
            "total_request_seconds": sum(durations), "gross_upper_microusd": total_cost}
save_private_json(folder / "results.json", {"contract": contract, "answers": answers,
    "metrics": metrics, "original_reference_metrics": original_metrics, "operations": operations})
show_table(["Model", "Rubric", "Valid responses", "Median request seconds", "Conservative USD estimate"],
    [(m, a, operations[m][a]["statuses"].get("ok", 0), round(operations[m][a]["median_seconds"], 2),
      round(operations[m][a]["gross_upper_microusd"] / 1_000_000, 4)) for m in models for a in arms])
print("Private results saved. Read the public summary for the interpreted outcome and next step.")


## 6. What we learned

**Rubric v4 improved both models on these 54 reviewed development cases. Both now meet the three quality targets on this sample.** Gemini 3.8 Flash reaches **87.1% precision, 83.1% recall and 98.1% coverage**; Qwen3.6 reaches **82.0%, 82.0% and 100%**. F1 rises by **9.3 percentage points for Gemini** and **6.4 points for Qwen** against fresh v3 controls scored with the same answers.

**What improved:** Gemini recovered six more politics tags, three more LGBTQ tags and two more human-rights tags. Qwen recovered five more culture tags and three more politics tags. These gains are consistent with the revised boundaries, but this combined experiment cannot identify the effect of each individual rule.

**What still needs care:** Qwen lost three previously matched social-issues tags; Gemini lost one, on a case with a retained interpretation limitation. One Qwen housing loss swaps housing for another accepted topic in a three-tag reference under the two-tag limit. Broader politics rules also add extra suggestions: three more for Qwen and two more for Gemini. Keep those tradeoffs visible. We did not change reference answers to remove the disagreements.

**Reliability:** 215 of 216 scheduled responses were valid. One Gemini v3 response exhausted its output allowance and counts as no tags. On the same 53 cases with valid answers in both Gemini arms, F1 still improves from **76.3% to 85.4% (+9.1 points)**. The overall gain is not explained by that single failure. All 108 Qwen responses and all 54 Gemini v4 responses were valid.

**Recommendation:** retain v4 for both as development candidates, with Gemini leading on agreement and Qwen offering a fast, free hosted alternative. Complete-tag agreement is still only **35/54 for Gemini and 29/54 for Qwen**, so these results support review, not automatic approval. Stop tuning this familiar set for now. Finish DR-24 accounting and use DR-27 to decide a small reviewed prototype or a separate confirmation on other manually reviewed cases. No fresh labeling or extra run is required to review this result; any later comparison must respect the reviewed-only requirement and be described honestly if its cases were previously examined.


Read the [aggregate comparison](../docs/results/rubric-v4-comparison.md) for per-tag changes, reference sensitivity and costs. These are familiar development cases, not independent confirmation. No reference labels changed during the run.
